# Lesson 9: Closed loop

**You will learn:** how a subject acts in an environment, where each action changes the next observation.
**Requires:** CPU only, no downloads, under a minute.

In the earlier lessons the inputs were fixed in advance. In a closed-loop task, such as a game or a robot task, the subject's action decides what it sees next. Brain-Score uses the same session idea:

| Object | Meaning |
| --- | --- |
| Environment | Your simulator or game, with `reset()` and `step(outputs)`. |
| Row | One step: a mapping of channel to payload, such as `{'observation': ..., 'reward': 0.0}`. |
| `environment_session` | Builds a session whose next input depends on the last response. |
| `policy` | The `BrainScoreModel` part that maps a row to a mapping of output channel to payload, such as `{'motor': action}`. |

## Setup

Imports and settings.

In [ ]:
import numpy as np

## The subject

A policy is a plain function from a row to a mapping of output channel to payload. This one always moves right; a real policy would be a trained network or a robot controller. Given as `policy=`, it makes the subject take observations in and give motor actions out.

In [ ]:
from brainscore import BrainScoreModel

def move_right(row):
    return {'motor': np.array(1)}      # +1 = one cell to the right

agent = BrainScoreModel(identifier='move-right', policy=move_right)
print('inputs: ', agent.in_channels)
print('outputs:', agent.out_channels)

## The experiment

### Environment

A five-cell corridor. The agent starts at cell 0; reaching cell 4 ends the episode with a reward of 1.

In [ ]:
class Corridor:
    """A five-cell corridor. The agent starts at 0 and the goal is cell 4."""

    def reset(self, seed=None, options=None):
        self.position = 0
        return {'observation': {'position': 0}, 'reward': 0.0}

    def step(self, outputs):
        self.position = max(0, self.position + int(outputs['motor']))     # the left wall stops at 0
        done = self.position >= 4
        row = {'observation': {'position': self.position}, 'reward': float(done)}
        return row, ('terminal' if done else None)       # None: the episode goes on

    def close(self):
        pass

### Task

One episode: reach the goal. There is no stimulus table; each observation is produced by the environment from the previous action.

## Run

`environment_session` resets the environment, delivers each row, applies each action, and stops when the episode ends. The subject participates exactly as in Lesson 1.

In [ ]:
from brainscore_core.streaming_helpers import environment_session

with environment_session(Corridor()) as session:
    agent.participate(session)
    complete = session.complete
    observations = [e for e in session.source.input_events if e.channel == 'observation']   # what the agent saw, in order
    actions = session.collect('motor')              # what it did after each observation
agent.reset()

## Read results back

The transcript pairs each observation with the action that followed it.

In [ ]:
print('episode complete:', complete)
for observation, action in zip(observations, actions):
    print(observation.payload, '->', action.payload)

Four steps to the right reach the goal. The `run_environment(subject, environment)` helper does the same in one call and returns the applied actions.

## Summary

- A closed-loop task is still a session; only the source of the inputs changes.
- The `policy` part maps each row to a mapping of output channel to payload.
- Existing robotics evaluators (for example LIBERO or DROID) can keep their own loop; see the docs.

**Next:** [Lesson 10](10_run_an_experiment.ipynb) organizes conditions, repetitions, recording and perturbation into a reproducible experiment.